# 21 - קריטיות משוקללת בביקוש (Demand-Weighted Criticality)

כל דירוג קריטיות שהופק עד כה בפרויקט זה הוא **טופולוגי בלבד**: תחנה חשובה בשל מיקומה בגרף, ולא בשל מספר האנשים שהיו נותרים תקועים בפועל אילו הייתה כושלת. מחברת זו בוחנת את כיוון ההמשך השני של הדוח - *שקלול הקריטיות לפי ביקוש נוסעים במקום לפי טופולוגיה בלבד* - ושואלת האם התשובה לשאלה "אילו תחנות קריטיות" משתנה כאשר מחזירים את האנשים לתוך הרשת.

## יש לקרוא זאת לפני קריאת כל מספר שלהלן

**קובץ ה-GTFS אינו מכיל נתוני נסיעות (ridership) כלשהם.** אין בו עליות, אין ירידות, אין תיקופי כרטיסים, אין מקדמי עומס ואין מטריצת מוצא-יעד באף קובץ של הפיד הישראלי. לפיכך, כל מה שמכונה במחברת זו *ביקוש* הוא **PROXY** (מדד מקורב), הבנוי משני דברים שכן קיימים ברשותנו:

1. **אוכלוסייה מגורית** לפי אזור סטטיסטי של הלמ"ס 2021, המשויכת לתחנות במחברת `08_socioeconomic_equity`; וכן
2. **נפח שירות** (`stop_use_count`, מספר ביקורי הנסיעות המתוזמנים בתחנה), המשמש רק כמכפיל מתון של נטיית העלייה לרכב.

ה-proxy מסומן ככזה בכל מקום שבו הוא מופיע: בטקסט, בכותרת של כל איור, ובשמות עמודות הפלט (`population_served`, `demand_weight`, `demand_weighted_betweenness_proxy`). סעיף 15 מפרט בהרחבה את הדרכים שבהן ה-proxy שגוי. אין לקרוא דבר מכאן כאומדן של מספר נוסעים בפועל.

## שאלת המחקר

אם משקללים קריטיות מבוססת מסלולים קצרים ביותר לפי מספר התושבים המתגוררים במוצא ובייעד של נסיעה, **אילו תחנות מרוויחות ואילו מאבדות חשיבות** ביחס לדירוג הטופולוגי הטהור של מחברת `04_centrality_analysis`? האם תחנה זניחה מבחינה מבנית בשכונה צפופה מדורגת גבוה יותר מתחנה מקושרת היטב באזור ריק?

## המתודולוגיה בפסקה אחת

אוכלוסיית כל אזור סטטיסטי של הלמ"ס מפוצלת בין התחנות הנמצאות בתוכו, ולאחר מכן מוקצית מחדש על פני תחנות שכנות באמצעות גרעין גרביטציה עם דעיכה לפי מרחק (כך שתושב מתחלק בין התחנות הנמצאות בפועל בטווח הליכה, והסך הארצי נשמר). כך מתקבל `population_served` לכל תחנה. הכפלה באיבר נפח השירות מניבה את `demand_weight`. לאחר מכן אנו מריצים אומדן של **betweenness משוקלל בביקוש**: אלגוריתם Brandes שבו תחנות המקור נדגמות בהסתברות פרופורציונית למשקל הביקוש שלהן, וכל תחנת יעד תורמת את משקל הביקוש שלה לצבירת התלויות. התוצאה מעריכה את `sum_{s,t} w(s) w(t) sigma_st(v) / sigma_st` - קריטיות מבוססת מסלולים קצרים ביותר, שבה כל מסלול שווה למספר האנשים המצויים באופן סביר בשני קצותיו. אנו משווים אותה מול `approx_betweenness` של מחברת 04, ומול **בקרה במשקלים אחידים** המורצת דרך *אותו* אומדן, כך שההשוואה מבודדת את השפעת שקלול הביקוש ולא את השפעת החלפת האומדן.

## קלט (חייב להתקיים לפני הרצת מחברת זו)

* `outputs/nb/02_graph_construction/tables/nodes.csv`, `edges.csv` - גרף סמיכות הנסיעות (מחברת **02**).
* `outputs/nb/04_centrality_analysis/tables/stop_metrics.csv` - קו הבסיס הטופולוגי, ובפרט העמודה `approx_betweenness` (מחברת **04**).
* `outputs/nb/08_socioeconomic_equity/tables/stops_with_socioeconomic.csv` וכן `socioeconomic_neighborhood_access.csv` - השיוך לאזורים הסטטיסטיים של הלמ"ס, האוכלוסייה והאשכול החברתי-כלכלי (מחברת **08**).

לא נקרא כאן אף קובץ GTFS גולמי: הקובץ `stop_times.txt` (816 MB) **אינו** נדרש, משום שה-proxy של האוכלוסייה נבנה מפלט שלב 08 והגרף מפלט שלב 02.

## פלט (הכול תחת `outputs/nb/21_demand_weighted_criticality/`)

* `tables/demand_proxy.csv` - `stop_id, population_served, demand_weight` (קובץ החוזה).
* `tables/demand_weighted_criticality.csv` - `stop_id, stop_name, topological_rank, demand_weighted_rank, rank_shift` (קובץ החוזה).
* `demand_summary.json` - מספרי הכותרת, הפרמטרים וסטטיסטיקות ההתאמה.
* `tables/demand_proxy_detail.csv` - ה-proxy על כל עמודות הביניים שלו ואיכות השיוך ללמ"ס לכל תחנה.
* `tables/criticality_lens_detail.csv` - שלוש עמודות הציון ושלוש עמודות הדירוג לכל תחנה.
* `tables/rank_agreement.csv` - Spearman rho וחפיפת top-N בין העדשות.
* `tables/rank_movers.csv` - המרוויחות והמפסידות הגדולות ביותר בדירוג.
* `tables/rank_shift_by_socioeconomic_cluster.csv` - תזוזת דירוג ממוצעת לכל אשכול למ"ס 1-10.
* `tables/proxy_sensitivity.csv`, `tables/betweenness_sensitivity.csv` - מידת ההשפעה של פרמטרי ה-proxy.
* `figures/*.png`.

דבר מחוץ לתיקיית שלב זה אינו נכתב. `outputs/tables`, `outputs/figures` ו-`outputs/rail` מכילים את התוצאות המצוטטות בדוח הכתוב ואינם נוגעים בהם כלל.

## 1. אתחול סביבת העבודה

התא שלהלן הוא ה-bootstrap הסטנדרטי של הפרויקט, זהה לזה שבמחברת 03, כך שמחברת זו רצה הן על checkout מקומי והן על Google Colab. `_ensure(...)` מתקין ב-pip רק חבילות שחסרות באמת (הרצה חוזרת זולה), ו-`find_repo_root()` מטפס כלפי מעלה מתיקיית העבודה בחיפוש אחר תיקיית ה-GTFS, ומשכפל את המאגר אל `/content` אם אינה נמצאת (מקרה Colab). הוא מגדיר את `REPO`, `DATA` ו-`OUT`, שכל תא מאוחר יותר תלוי בהם, ולכן עליו לרוץ ראשון.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. ספריות, תיקיות השלב וקבועים הניתנים לכוונון

שלב זה מחזיק בתיקיית פלט אחת בדיוק, `outputs/nb/21_demand_weighted_criticality/`, ובה תת-התיקיות `tables/` ו-`figures/`, בהתאם למוסכמת הפרויקט.

כל זמן הריצה וכל בחירות המידול מרוכזים בקבועים שלהלן, כך שהבודק יכול לשנות את המודל או להמיר זמן ריצה בדיוק במקום אחד:

* `CATCHMENT_RADIUS_M = 500` - טווח ההליכה (catchment) של תחנה, במטרים. תושבים הרחוקים מכך אינם מוקצים אליה. 500 מ' הוא ה-catchment המקובל לתחנת אוטובוס; 800 מ' משמש לעיתים קרובות לרכבת.
* `DECAY_LENGTH_M = 250` - גרעין הגרביטציה הוא `exp(-d / DECAY_LENGTH_M)`, כך שתחנה במרחק 250 מ' מקבלת `1/e` מעוצמת המשיכה של תחנה בפתח הבית.
* `SERVICE_EXPONENT = 0.5` - עד כמה חזק נפח השירות מכפיל את איבר האוכלוסייה (`0` = אוכלוסייה בלבד, `1` = פרופורציונלי לחלוטין לביקורים המתוזמנים). הרצת רגישות בסעיף 14 מראה מה עולה בחירה זו.
* `K_DEMAND_SOURCES = 400` - מספר המקורות הנדגמים לכל עדשת betweenness. **זהו כפתור העלות המרכזי.** כל מקור הוא BFS אחד בתוספת צבירת תלויות אחת על פני הרכיב הגדול ביותר בן ~30k צמתים, כ-0.2-0.5 שנ' ב-Python טהור, ואנו מריצים **שתי** עדשות (משוקללת בביקוש והבקרה האחידה), ולכן יש לצפות לכ-3-8 דקות כאן. ניתן להנמיך ל-150 למעבר מהיר; האומדן חסר הטיה בכל `k`, רק רועש יותר.
* `RUN_WEIGHT_SENSITIVITY` / `RUN_BETWEENNESS_SENSITIVITY` - שני בלוקי הרגישות. הראשון זול (כמה שניות לכל רדיוס catchment). השני מריץ מחדש את אומדן ה-betweenness עם `SENSITIVITY_K = 150` מקורות עבור שתי הגדרות proxy חלופיות, ומוסיף עוד כמה דקות; ניתן להגדירו כ-`False` אם ממהרים, אך סעיף 14 הוא חלק מטיעון היושרה המתודולוגית.

אומדן ה-betweenness כתוב במלואו בסעיף 9 ולא נלקח מ-`networkx`, משום ש-`networkx.betweenness_centrality` אינו יכול לשקלל את נקודות הקצה - וזוהי בדיוק כל מטרתה של מחברת זו.

In [ ]:
# --- Libraries, stage folders and every tunable constant -------------------
_ensure('pandas', 'numpy', 'networkx', 'matplotlib', 'seaborn', 'scipy')

import json
import math
import time
from collections import deque

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.05)

STAGE = OUT / '21_demand_weighted_criticality'
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Demand PROXY parameters ----------------------------------------------
CATCHMENT_RADIUS_M = 500.0    # walking catchment of a stop, metres
DECAY_LENGTH_M = 250.0        # gravity kernel exp(-d / DECAY_LENGTH_M)
SERVICE_EXPONENT = 0.5        # 0 = population only, 1 = fully service-proportional

# --- Betweenness estimator cost -------------------------------------------
K_DEMAND_SOURCES = 400        # sampled sources per lens (main cost knob)
SEED_DEMAND = 21              # seed of the demand-weighted lens
SEED_UNIFORM = 22             # seed of the uniform-weight control lens
PROGRESS_EVERY = 50           # print a progress line every N sampled sources

# --- Sensitivity blocks ----------------------------------------------------
RUN_WEIGHT_SENSITIVITY = True        # cheap: re-computes the proxy only
RUN_BETWEENNESS_SENSITIVITY = True   # costly: two extra estimator runs
SENSITIVITY_K = 150                  # sources used inside the sensitivity runs
RADIUS_GRID = [300.0, 500.0, 1000.0]
SERVICE_EXPONENT_GRID = [0.0, 0.5, 1.0]

# --- Presentation ----------------------------------------------------------
TOP_N = 15                    # rows in the top-N tables / bars in the charts
MOVERS_N = 25                 # gainers and losers written to rank_movers.csv
OVERLAP_SIZES = [50, 200, 1000]
FIG_DPI = 150

print('stage folder :', STAGE)
print('networkx', nx.__version__, '| pandas', pd.__version__, '| numpy', np.__version__)

## 3. רינדור תוויות בעברית

שמות התחנות בפיד ה-GTFS הישראלי הם בעברית, וכמה איורים שלהלן מדפיסים אותם (בפרט תרשימי תזוזות הדירוג - כל העניין הוא *אילו* תחנות זזות). Matplotlib אינו מממש את אלגוריתם הדו-כיווניות של Unicode, ולכן טקסט מימין לשמאל מרונדר הפוך. התא שלהלן הוא ה-monkey-patch החד-פעמי הסטנדרטי של הפרויקט ל-`matplotlib.text.Text.set_text`: כל מחרוזת המכילה תווים עבריים מומרת לסדר תצוגה באמצעות `python-bidi` לפני הציור, ונבחר גופן בעל גליפים עבריים (Arial ב-Windows, DejaVu Sans במקומות אחרים). הפעולה אידמפוטנטית. כל שאר הטקסט במחברת זו הוא באנגלית.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. איתור השלבים הקודמים

מחברת זו צורכת שלושה שלבים קודמים. במקום לקבע שמות תיקיות בקוד, `stage_dir` מאתר שלב לפי **קידומת בת שתי ספרות** (`OUT.glob('04*')`), כך שהוא ממשיך לעבוד גם אם תיקיית שלב משנה את שמה מ-`04_centrality_analysis` לכל שם אחר המתחיל ב-`04`. לאחר מכן `load_stage_table` מחפש בתיקייה זו רקורסיבית את הקובץ, כך שלא משנה אם הטבלה יושבת בשורש התיקייה או בתוך `tables/`.

שני העוזרים מעלים `FileNotFoundError` המציין **איזו מחברת יש להריץ תחילה**, משום שחוסר בפריט מוצר משלב קודם הוא הסיבה הסבירה ביותר לכישלון מחברת זו אצל אדם אחר.

In [ ]:
# --- Resolve earlier stages by two-digit prefix ----------------------------
def stage_dir(prefix, notebook_hint):
    """Return the stage folder whose name starts with `prefix`, else raise."""
    matches = sorted(p for p in OUT.glob(prefix + '*') if p.is_dir())
    if not matches:
        raise FileNotFoundError(
            f'No stage folder starting with "{prefix}" under {OUT} - '
            f'run notebook {notebook_hint} first.'
        )
    return matches[0]


def load_stage_table(prefix, filename, notebook_hint, **read_kwargs):
    """Load a CSV written by an earlier stage, with an actionable error message."""
    folder = stage_dir(prefix, notebook_hint)
    candidates = [folder / 'tables' / filename, folder / filename]
    path = next((c for c in candidates if c.exists()), None)
    if path is None:
        found = sorted(folder.rglob(filename))
        path = found[0] if found else None
    if path is None:
        raise FileNotFoundError(
            f'"{filename}" not found anywhere under {folder} - '
            f'run notebook {notebook_hint} first; it is the notebook that writes it.'
        )
    frame = pd.read_csv(path, encoding='utf-8-sig', **read_kwargs)
    print(f'{filename:<42} {len(frame):>7,} rows  <-  {path}')
    return frame


def require_columns(frame, columns, filename):
    """Fail loudly if an upstream contract column is missing."""
    missing = [c for c in columns if c not in frame.columns]
    if missing:
        raise KeyError(
            f'{filename} is missing the column(s) {missing}. '
            f'It has {list(frame.columns)}. Re-run the notebook that produces it.'
        )


print('helpers ready')

## 5. טעינת הקלט

נקראות חמש טבלאות:

* `nodes.csv` / `edges.csv` (שלב 02) - גרף סמיכות הנסיעות ותכונות התחנות, ובכללן `stop_use_count`, מספר ביקורי הנסיעות המתוזמנים בכל תחנה, המהווה את איבר נפח השירות שלנו.
* `stop_metrics.csv` (שלב 04) - קו הבסיס הטופולוגי. העמודה שמולה אנו משווים היא `approx_betweenness`; יש לשים לב שהיא עצמה אומדן מבוסס דגימת `k` (מחברת 04 השתמשה ב-300 מקורות), ולכן היא נושאת עמה רעש דגימה משלה אל כל השוואה שלהלן.
* `stops_with_socioeconomic.csv` (שלב 08) - האזור הסטטיסטי של הלמ"ס לכל תחנה (`socio_unit_id`), אוכלוסיית האזור (`socio_population`), האשכול החברתי-כלכלי שלו 1-10 (`socio_cluster`) וחשוב מכך, **כיצד בוצע השיוך** (`socio_join_method` = `within` להתאמת נקודה-בתוך-פוליגון, `nearest` לנפילה חזרה לפוליגון הקרוב ביותר, בתוספת `socio_join_distance_m`).
* `socioeconomic_neighborhood_access.csv` (שלב 08) - אותם אזורים בצורה מצטברת, המשמשים להשלמת אוכלוסיית אזור החסרה בטבלה ברמת התחנה ולבדיקה צולבת של מספר התחנות באזור.

אנו מוודאים את עמודות החוזה מיד, כך ששינוי סכמה בשלב קודם ייכשל כאן ולא יפיק דירוג שגוי בשקט.

In [ ]:
# --- Load the artifacts of notebooks 02, 04 and 08 -------------------------
nodes_df = load_stage_table('02', 'nodes.csv', '02_graph_construction',
                            dtype={'stop_id': str})
edges_df = load_stage_table('02', 'edges.csv', '02_graph_construction',
                            dtype={'from_stop': str, 'to_stop': str})
metrics_df = load_stage_table('04', 'stop_metrics.csv', '04_centrality_analysis',
                              dtype={'stop_id': str})
socio_df = load_stage_table('08', 'stops_with_socioeconomic.csv',
                            '08_socioeconomic_equity', dtype={'stop_id': str})
access_df = load_stage_table('08', 'socioeconomic_neighborhood_access.csv',
                             '08_socioeconomic_equity')

require_columns(nodes_df, ['stop_id', 'stop_name', 'lat', 'lon', 'region', 'stop_use_count'],
                'nodes.csv')
require_columns(edges_df, ['from_stop', 'to_stop', 'trip_frequency'], 'edges.csv')
require_columns(metrics_df, ['stop_id', 'approx_betweenness'], 'stop_metrics.csv')
require_columns(socio_df, ['stop_id', 'socio_unit_id', 'socio_population', 'socio_cluster'],
                'stops_with_socioeconomic.csv')
require_columns(access_df, ['socio_unit_id', 'population', 'stops'],
                'socioeconomic_neighborhood_access.csv')

nodes_df['stop_id'] = nodes_df['stop_id'].astype(str)
metrics_df['stop_id'] = metrics_df['stop_id'].astype(str)
socio_df['stop_id'] = socio_df['stop_id'].astype(str)

print()
print(f'stops in the graph          : {nodes_df["stop_id"].nunique():,}')
print(f'stops with a CBS join       : {socio_df["socio_unit_id"].notna().sum():,}')
print(f'CBS statistical areas        : {access_df["socio_unit_id"].nunique():,}')
print(f'stops covered by stage 04    : {metrics_df["stop_id"].nunique():,}')

## 6. בנייה מחדש של הגרף הלא-מכוון והרכיב הגדול ביותר שלו

הקריטיות כאן היא גודל מבוסס מסלולים קצרים ביותר, ולכן, כמו בכל מקום אחר בפרויקט, היא מחושבת על **ההיטל הלא-מכוון** `G` של גרף הנסיעות (קשת אחת לכל זוג תחנות בלתי מסודר, כאשר תדירויות הנסיעה של שני הכיוונים מסוכמות) ומוגבלת אל **הרכיב הקשיר הגדול ביותר** `Gc`. תחנות מחוץ לרכיב הגדול ביותר אינן יכולות לשכון על מסלול קצר ביותר בתוכו, ולכן ציונן הוא 0 בכל עדשה - זהו הערך הכן, ולא ערך חסר.

מסלולים קצרים ביותר נספרים ב-**קפיצות (hops)** (מספר המקטעים המזערי), בדיוק כמו במחברת 04, כך שקו הבסיס הטופולוגי והעדשה המשוקללת בביקוש נבדלים *רק* באופן שבו נקודות הקצה משוקללות. מסלולים משוקללים בזמן נסיעה היו בחירת מידול שונה (ולגיטימית); ערבוב שני השינויים בבת אחת היה הופך את ההשוואה לבלתי ניתנת לפרשנות.

In [ ]:
# --- Undirected projection and largest connected component -----------------
def build_undirected_graph(nodes_frame, edges_frame):
    """One edge per unordered station pair; weights of both directions summed."""
    graph = nx.Graph()
    graph.add_nodes_from(nodes_frame['stop_id'].astype(str))
    freq = pd.to_numeric(edges_frame['trip_frequency'], errors='coerce').fillna(1.0)
    for u, v, w in zip(edges_frame['from_stop'].astype(str),
                       edges_frame['to_stop'].astype(str),
                       freq.astype(float)):
        if u == v:
            continue
        if graph.has_edge(u, v):
            graph[u][v]['weight'] += w
        else:
            graph.add_edge(u, v, weight=w)
    return graph


G = build_undirected_graph(nodes_df, edges_df)
components = sorted(nx.connected_components(G), key=len, reverse=True)
Gc = G.subgraph(components[0]).copy()
LCC_NODES = set(Gc.nodes)

print(f'undirected graph      : {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
print(f'connected components  : {len(components):,}')
print(f'largest component     : {Gc.number_of_nodes():,} nodes '
      f'({Gc.number_of_nodes() / G.number_of_nodes():.1%}), {Gc.number_of_edges():,} edges')
print(f'outside the LCC       : {G.number_of_nodes() - Gc.number_of_nodes():,} stations '
      f'(scored 0 in every lens)')

## 7. שלב 1 של ה-proxy: פיצול אוכלוסיית כל אזור סטטיסטי בין תחנותיו

הלמ"ס מספקת אוכלוסייה לכל **אזור סטטיסטי** (פוליגון בגודל שכונה), ולא לכל תחנה. לפיכך הצעד הראשון הוא הקצאה נאיבית אך שקופה: **תושבי כל אזור מחולקים שווה בשווה בין התחנות המשויכות לאותו אזור**. אזור עם 6,000 תושבים ו-20 תחנות מעניק לכל אחת מהתחנות הללו מנת בסיס של 300 תושבים.

זוהי בחירה גסה במכוון, והיא גסה בכיוון מסוים: היא מניחה שהתושבים פרושים באופן אחיד על פני האזור ומשתמשים בתחנות שבו באופן אחיד. משמעות נוספת היא ש**אזורים צפופי תחנות מדללים את עצמם** - שתי תחנות במרחק 80 מ' זו מזו בצדדים מנוגדים של אותו רחוב מקבלות כל אחת מחצית מהמנה, וזה נכון בקירוב עבור proxy אך שגוי בפרטים (מבחינת הנוסע מדובר באותו זוג תחנות).

שתי עובדות של איכות נתונים משלב 08 מועברות הלאה ומדווחות כאן במקום להיות מוסתרות:

* לא כל תחנה נופלת בתוך פוליגון. שלב 08 נופל חזרה לפוליגון **הקרוב ביותר** (`socio_join_method = 'nearest'`, עם המרחק ב-`socio_join_distance_m`). תחנה שהותאמה לאזור במרחק 2 ק"מ מזוכה באנשים שאינם מתגוררים בקרבתה כלל.
* לכמה תחנות **אין יחידת למ"ס כלל**, ולכמה יחידות **אין נתון אוכלוסייה**. תחנות אלו מקבלות מנת בסיס 0, כלומר הן בלתי נראות כמוצא/יעד בעדשת הביקוש. אנו סופרים אותן במפורש.

במקומות שבהם לטבלה ברמת התחנה אין אוכלוסייה עבור יחידה, אנו נופלים חזרה לאוכלוסייה הרשומה עבור אותה יחידה ב-`socioeconomic_neighborhood_access.csv`.

In [ ]:
# --- Step 1: population of a statistical area, split across its stops ------
socio = socio_df.drop_duplicates(subset='stop_id').copy()

unit_pop_from_stops = (socio.dropna(subset=['socio_unit_id'])
                            .groupby('socio_unit_id')['socio_population'].first())
unit_pop_from_access = (access_df.dropna(subset=['socio_unit_id'])
                                 .drop_duplicates(subset='socio_unit_id')
                                 .set_index('socio_unit_id')['population'])
unit_population = unit_pop_from_stops.combine_first(unit_pop_from_access)

stops_per_unit = (socio.dropna(subset=['socio_unit_id'])
                       .groupby('socio_unit_id')['stop_id'].nunique())

join_cols = ['stop_id', 'socio_unit_id', 'socio_cluster',
             'socio_join_method', 'socio_join_distance_m']
join_cols = [c for c in join_cols if c in socio.columns]

proxy = (nodes_df[['stop_id', 'stop_name', 'lat', 'lon', 'region', 'metro', 'stop_use_count']]
         .drop_duplicates(subset='stop_id')
         .merge(socio[join_cols], on='stop_id', how='left'))

proxy['unit_population'] = proxy['socio_unit_id'].map(unit_population)
proxy['unit_stops'] = proxy['socio_unit_id'].map(stops_per_unit)
proxy['area_population_share'] = (
    proxy['unit_population'] / proxy['unit_stops'].replace(0, np.nan)
).fillna(0.0)

national_population = float(unit_population.dropna().sum())
allocated = float(proxy['area_population_share'].sum())
no_unit = int(proxy['socio_unit_id'].isna().sum())
no_population = int((proxy['socio_unit_id'].notna() & proxy['unit_population'].isna()).sum())

print(f'CBS population across all areas   : {national_population:,.0f}')
print(f'population allocated to stops     : {allocated:,.0f} '
      f'({allocated / national_population:.1%} of it)')
print(f'stops with no CBS unit            : {no_unit:,}')
print(f'stops whose unit has no population: {no_population:,}')
if 'socio_join_method' in proxy.columns:
    print('\njoin method used per stop:')
    print(proxy['socio_join_method'].value_counts(dropna=False).to_string())
if 'socio_join_distance_m' in proxy.columns:
    far = int((proxy['socio_join_distance_m'] > 1000).sum())
    print(f'\nstops joined to an area more than 1 km away: {far:,} '
          f'(their population attribution is weak)')
proxy[['stop_id', 'stop_name', 'socio_unit_id', 'unit_population',
       'unit_stops', 'area_population_share']].head()

## 8. שלב 2 של ה-proxy: גרעין גרביטציה על פני מרחק הליכה

פיצול אוכלוסיית אזור לפי השתייכות לאזור בלבד יוצר אי-רציפות בגבול הפוליגון: תושב במרחק 30 מ' מתחנה מזוכה לה רק אם במקרה הוא נופל באותו צד של קו מנהלי. הצעד השני מחליק זאת באמצעות **מודל גרביטציה**.

כל תחנה `j` מפזרת מחדש את מנת הבסיס שלה `p_j` על פני כל התחנות שבמרחק של עד `CATCHMENT_RADIUS_M` מטרים ממנה (כולל היא עצמה), באופן פרופורציוני ל-`exp(-d / DECAY_LENGTH_M)`. המשקלים מנורמלים **לכל מקור**, ולכן:

```
population_served[i] = sum_j p_j * exp(-d_ij / L) / sum_{k in radius(j)} exp(-d_jk / L)
```

מכיוון שהגרעין מנורמל לכל מקור, הסך הארצי **נשמר** - אף תושב אינו נספר פעמיים, וזהו הבאג הרגיל בספירות catchment מבוססות רדיוס ("אוכלוסייה בטווח 500 מ' מכל תחנה", בסכימה על פני התחנות, עלולה לעלות על אוכלוסיית המדינה פי כמה). מה שהגרעין עושה במקום זאת הוא לרכז ביקוש במקומות שבהם התחנות מקובצות: תחנה בודדת המשרתת אזור שלם שומרת את כולו, בעוד שאחת מתוך עשרים תחנות ברשת צפופה שומרת חלק משלה ואוספת שברים משכנותיה.

המרחקים משתמשים בקירוב equirectangular סביב קו הרוחב הממוצע של הפיד, המדויק הרבה מתחת למטר בסדרי גודל אלה, והתחנות משובצות לרשת של תאים בגודל `CATCHMENT_RADIUS_M`, כך שנסרקת רק סביבת ה-3x3 של התאים. העלות: מכמה שניות ועד דקה עבור ~30k תחנות. תחנות עם קואורדינטות חסרות שומרות את מנת הבסיס שלהן ואינן משתתפות בשום חילופין.

In [ ]:
# --- Step 2: distance-decayed re-allocation (population-conserving) --------
def allocate_population(lat, lon, base_pop, radius_m, decay_m):
    """Spread each stop's base population over stops within `radius_m`.

    Weights are exp(-d / decay_m), normalised per source, so the sum of the
    returned array equals the sum of `base_pop` (no double counting).
    """
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    base_pop = np.asarray(base_pop, dtype=float)
    n = len(lat)
    served = np.zeros(n, dtype=float)

    valid = np.isfinite(lat) & np.isfinite(lon)
    m_per_deg_lat = 110574.0
    m_per_deg_lon = 111320.0 * math.cos(math.radians(float(np.nanmean(lat[valid]))))

    x = lon * m_per_deg_lon
    y = lat * m_per_deg_lat

    # bucket stops into square cells of side `radius_m`
    cells = {}
    ci = np.zeros(n, dtype=np.int64)
    cj = np.zeros(n, dtype=np.int64)
    for idx in np.flatnonzero(valid):
        ci[idx] = int(math.floor(y[idx] / radius_m))
        cj[idx] = int(math.floor(x[idx] / radius_m))
        cells.setdefault((ci[idx], cj[idx]), []).append(idx)
    cells = {key: np.asarray(val, dtype=np.int64) for key, val in cells.items()}

    neighbour_cache = {}

    def candidates(key):
        got = neighbour_cache.get(key)
        if got is None:
            parts = [cells[(key[0] + a, key[1] + b)]
                     for a in (-1, 0, 1) for b in (-1, 0, 1)
                     if (key[0] + a, key[1] + b) in cells]
            got = np.concatenate(parts) if parts else np.empty(0, dtype=np.int64)
            neighbour_cache[key] = got
        return got

    for idx in np.flatnonzero(base_pop > 0):
        if not valid[idx]:
            served[idx] += base_pop[idx]      # no coordinates: keep it in place
            continue
        cand = candidates((ci[idx], cj[idx]))
        d = np.hypot(x[cand] - x[idx], y[cand] - y[idx])
        keep = d <= radius_m
        cand, d = cand[keep], d[keep]
        if cand.size == 0:
            served[idx] += base_pop[idx]
            continue
        w = np.exp(-d / decay_m)
        total = w.sum()
        if not np.isfinite(total) or total <= 0:
            served[idx] += base_pop[idx]
            continue
        np.add.at(served, cand, base_pop[idx] * (w / total))
    return served


t0 = time.time()
proxy['population_served'] = allocate_population(
    proxy['lat'].to_numpy(), proxy['lon'].to_numpy(),
    proxy['area_population_share'].to_numpy(),
    CATCHMENT_RADIUS_M, DECAY_LENGTH_M,
)
print(f'gravity allocation (radius={CATCHMENT_RADIUS_M:.0f} m, '
      f'decay={DECAY_LENGTH_M:.0f} m) in {time.time() - t0:.1f}s')
print(f'conservation check : allocated {proxy["area_population_share"].sum():,.0f} -> '
      f'served {proxy["population_served"].sum():,.0f}')
print(f'stops with zero population PROXY: '
      f'{int((proxy["population_served"] <= 0).sum()):,} of {len(proxy):,}')
print(proxy['population_served'].describe().round(1).to_string())

## 9. שלב 3 של ה-proxy: שילוב נפח השירות, וטבלת החוזה

אוכלוסייה לבדה מעידה כמה אנשים גרים ליד תחנה, ולא כמה מהם יכולים באופן סביר *להשתמש* בה. תחנה המשורתת בארבע נסיעות ביום ותחנה המשורתת בארבע מאות אינן מושכות את אותו חלק מהנסיעות של שכונתן. לפיכך אנו מכפילים באיבר שירות:

```
demand_weight_raw[i] = population_served[i] * (stop_use_count[i] / median_stop_use_count) ** SERVICE_EXPONENT
demand_weight[i]     = demand_weight_raw[i] / sum(demand_weight_raw)      # sums to 1
```

כאשר `SERVICE_EXPONENT = 0.5` כברירת מחדל - שילוב מסוג שורש ריבועי, שנבחר כך שנפח השירות יווסת את ה-proxy מבלי לשלוט בו.

**צעד זה הוא פשרה, והוא חותך לשני הכיוונים.** הכללת נפח השירות הופכת את ה-proxy למציאותי יותר (אנשים אכן משתמשים בתחנות שיש בהן אוטובוסים) אך מייבאת חזרה חלקית את אותה טופולוגיה שאותה אנו מנסים לשקלל *הצידה*: `stop_use_count` קשור קשר חזק לדרגה משוקללת. לפיכך סעיף 14 חוזר על כל הניתוח עם `SERVICE_EXPONENT = 0` (אוכלוסייה טהורה, בלתי תלויה לחלוטין בשירות), כדי שהקורא יוכל לראות בדיוק כמה מהתוצאה נובע מאות האוכלוסייה וכמה מאות השירות.

תא זה כותב את קובץ החוזה הראשון, `tables/demand_proxy.csv`, ובו בדיוק שלוש העמודות המוסכמות `stop_id, population_served, demand_weight`. כל עמודת ביניים, בתוספת איכות השיוך ללמ"ס לכל תחנה, נכתבת אל `tables/demand_proxy_detail.csv` כך שדבר אינו אובד.

In [ ]:
# --- Step 3: demand weight = population PROXY x service volume -------------
def build_demand_weight(population_served, stop_use_count, service_exponent):
    """Return weights summing to 1 (or all-zero if no population is available)."""
    pop = np.asarray(population_served, dtype=float)
    use = pd.to_numeric(pd.Series(stop_use_count), errors='coerce').fillna(0.0).to_numpy()
    positive = use[use > 0]
    reference = float(np.median(positive)) if positive.size else 1.0
    relative = np.where(use > 0, use / reference, 1e-6)
    raw = pop * np.power(relative, service_exponent)
    raw = np.where(np.isfinite(raw) & (raw > 0), raw, 0.0)
    total = raw.sum()
    return raw / total if total > 0 else raw


proxy['demand_weight'] = build_demand_weight(
    proxy['population_served'], proxy['stop_use_count'], SERVICE_EXPONENT)

# --- contract table: exactly the three agreed columns ----------------------
(proxy[['stop_id', 'population_served', 'demand_weight']]
    .sort_values('demand_weight', ascending=False)
    .to_csv(TABLES / 'demand_proxy.csv', index=False, encoding='utf-8-sig'))

detail_cols = [c for c in ['stop_id', 'stop_name', 'lat', 'lon', 'region', 'metro',
                           'socio_unit_id', 'socio_cluster', 'socio_join_method',
                           'socio_join_distance_m', 'unit_population', 'unit_stops',
                           'area_population_share', 'stop_use_count',
                           'population_served', 'demand_weight'] if c in proxy.columns]
(proxy[detail_cols]
    .sort_values('demand_weight', ascending=False)
    .to_csv(TABLES / 'demand_proxy_detail.csv', index=False, encoding='utf-8-sig'))

share_top1pct = float(proxy['demand_weight'].nlargest(max(1, len(proxy) // 100)).sum())
rho_pop_service = float(pd.Series(proxy['population_served']).corr(
    pd.to_numeric(proxy['stop_use_count'], errors='coerce'), method='spearman'))

print(f'demand_proxy.csv written ({len(proxy):,} rows) -> {TABLES / "demand_proxy.csv"}')
print(f'top 1% of stops hold {share_top1pct:.1%} of the total PROXY demand weight')
print(f'Spearman(population PROXY, service volume) = {rho_pop_service:.3f}')
print(f'stops with zero PROXY demand weight: '
      f'{int((proxy["demand_weight"] <= 0).sum()):,} (invisible as origins/destinations)')
proxy.nlargest(TOP_N, 'demand_weight')[
    ['stop_name', 'region', 'socio_cluster', 'stop_use_count',
     'population_served', 'demand_weight']].round(5)

## 10. כיצד נראה ה-proxy

שלושה מבטים, כולם מסומנים כ-proxy:

1. **ההתפלגות** של `population_served`. זנב ימני ארוך הוא הצפוי - קומץ תחנות יושבות לבדן באזור צפוף ויורשות אלפי תושבים, בעוד שרובן חולקות את אזורן עם תריסר שכנות.
2. **נפח שירות מול ה-proxy של האוכלוסייה**. פיזור זה הוא בדיקת היושרה עבור סעיף 9: אילו השניים היו מתואמים כמעט לחלוטין, עדשת הביקוש הייתה דירוג שירות בשם אחר וכל התרגיל היה מעגלי. ערך ה-Spearman rho המודפס לעיל מלמד כמה אות בלתי תלוי מוסיף איבר האוכלוסייה בפועל.
3. **מפה** של ה-proxy, בסקאלת צבע לוגריתמית. היא אמורה להיראות כמו מפת אוכלוסייה של ישראל - צפופה לאורך מישור החוף ובירושלים, דלילה בנגב - ו*לא* כמו מפה של רשת האוטובוסים. אילו נראתה כמו האחרונה, ה-proxy היה מודד שירות ולא אנשים.

In [ ]:
# --- Figures: what the demand PROXY looks like -----------------------------
served = proxy['population_served'].astype(float)
use = pd.to_numeric(proxy['stop_use_count'], errors='coerce').fillna(0.0)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))
axes[0].hist(served[served > 0], bins=60, color='#0891b2')
axes[0].set_yscale('log')
axes[0].set_xlabel('PROXY population served (residents allocated per stop)')
axes[0].set_ylabel('Stations (log scale)')
axes[0].set_title('Population PROXY per stop (NOT ridership)')

axes[1].scatter(use.where(use > 0), served, s=4, alpha=0.25, color='#7c3aed')
axes[1].set_xscale('log')
axes[1].set_xlabel('Scheduled stop visits per stop (service volume)')
axes[1].set_ylabel('PROXY population served')
axes[1].set_title(f'Service volume vs population PROXY (Spearman rho = {rho_pop_service:.2f})')
fig.tight_layout()
fig.savefig(FIGURES / 'demand_proxy_distribution.png', dpi=FIG_DPI)
plt.show()

geo = proxy.dropna(subset=['lat', 'lon']).copy()
geo = geo[geo['demand_weight'] > 0]
if len(geo):
    fig, ax = plt.subplots(figsize=(7.5, 10))
    sc = ax.scatter(geo['lon'], geo['lat'], s=5,
                    c=np.log10(geo['demand_weight'] + 1e-12),
                    cmap='viridis', alpha=0.7)
    fig.colorbar(sc, ax=ax, shrink=0.7, label='log10 PROXY demand weight')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('PROXY demand weight per stop\n(population-based estimate, not measured ridership)')
    fig.tight_layout()
    fig.savefig(FIGURES / 'demand_proxy_map.png', dpi=FIG_DPI)
    plt.show()
else:
    print('No stop has both coordinates and a positive PROXY weight - skipping the map.')

## 11. אומדן ה-betweenness המשוקלל בביקוש

betweenness סטנדרטי מתייחס לכל זוג תחנות סדור כאל יחידת תנועה אחת. אנו רוצים

```
BC_demand(v) = sum over s != v != t of  w(s) * w(t) * sigma_st(v) / sigma_st
```

כאשר `w` הוא משקל הביקוש ה-PROXY: מסלול קצר ביותר שווה למספר האנשים המצויים באופן סביר בשני קצותיו. `networkx` אינו מסוגל לכך (ל-`betweenness_centrality` שלו אין שקלול נקודות קצה), ולכן התא שלהלן מממש את אלגוריתם Brandes ישירות, בשני שינויים:

* **שקלול יעדים.** בצבירה לאחור, הביטוי הרגיל `(1 + delta[w])` הופך ל-`(w(t) + delta[w])`, מה שגורם לכל יעד לתרום באופן פרופורציוני למשקל הביקוש שלו. זוהי ההכללה הקנונית של Brandes לנקודות קצה משוקללות והיא אינה עולה דבר.
* **דגימת מקורות פרופורציונית לביקוש.** סכימה על פני כל ~30k המקורות היא עבודה של שעות. במקום זאת אנו מגרילים `K_DEMAND_SOURCES` מקורות **עם החזרה, בהסתברות `w(s) / sum(w)`**, צוברים את התלויות, ומשנים קנה מידה לפי `sum(w) / k`. מכיוון שהסתברות הדגימה היא בדיוק משקל המקור, זהו אומדן חסר הטיה של הסכום לעיל - שקלול המקורות מתקבל "בחינם" מהתפלגות הדגימה, ואין צורך בתיקון importance-weighting.

הצבת `w = 1` בכל מקום מצמצמת את האומדן ל-betweenness נדגם רגיל (עד כדי קבוע `n`, שאינו משפיע על הדירוגים). כך אנו מקבלים את עדשת **הבקרה האחידה**: אותו קוד, אותו משטר seed, אותו מספר מקורות, כך שכל הבדל בין שתי העדשות נגרם משקלול הביקוש ומדבר זולתו.

**עלות.** BFS אחד בתוספת צבירה אחת על פני הרכיב הגדול ביותר לכל מקור נדגם: כ-0.2-0.5 שנ' לכל אחד ב-Python טהור, כלומר כ-1.5-4 דקות לכל עדשה ב-`k = 400`. האומדן חסר הטיה בכל `k`, אך ציונים בודדים - במיוחד בזנב - רועשים, בדיוק כפי ש-`approx_betweenness` של מחברת 04 רועש. דירוגים סמוך לצמרת יציבים בהרבה מדירוגים באמצע.

In [ ]:
# --- Brandes betweenness with demand-weighted endpoints, sampled sources ---
def sampled_endpoint_weighted_betweenness(graph, node_weight, k, seed,
                                          progress_every=None, label=''):
    """Estimate sum_{s,t} w(s) w(t) sigma_st(v) / sigma_st for every v.

    Sources are drawn with replacement with probability proportional to
    `node_weight`; targets contribute their own weight inside the Brandes
    accumulation. The estimate is unbiased for any k.
    """
    nodes = list(graph.nodes)
    weights = np.array([float(node_weight.get(n, 0.0)) for n in nodes], dtype=float)
    weights[~np.isfinite(weights)] = 0.0
    weights[weights < 0] = 0.0
    total_weight = float(weights.sum())
    if total_weight <= 0:
        raise ValueError('every node weight is zero - cannot sample sources')

    rng = np.random.default_rng(seed)
    draws = rng.choice(len(nodes), size=int(k), replace=True, p=weights / total_weight)
    target_weight = {n: float(weights[i]) for i, n in enumerate(nodes)}
    scores = dict.fromkeys(nodes, 0.0)

    t_start = time.time()
    for step, source_index in enumerate(draws, start=1):
        s = nodes[source_index]
        # ---- forward pass: BFS shortest-path DAG (hop distance) ----
        stack = []
        preds = {s: []}
        sigma = {s: 1.0}
        dist = {s: 0}
        queue = deque([s])
        while queue:
            v = queue.popleft()
            stack.append(v)
            dist_v = dist[v]
            sigma_v = sigma[v]
            for nbr in graph[v]:
                if nbr not in dist:
                    dist[nbr] = dist_v + 1
                    sigma[nbr] = 0.0
                    preds[nbr] = []
                    queue.append(nbr)
                if dist[nbr] == dist_v + 1:
                    sigma[nbr] += sigma_v
                    preds[nbr].append(v)
        # ---- backward pass: dependency accumulation, targets weighted ----
        delta = dict.fromkeys(stack, 0.0)
        while stack:
            w_node = stack.pop()
            coeff = (target_weight[w_node] + delta[w_node]) / sigma[w_node]
            for v in preds[w_node]:
                delta[v] += sigma[v] * coeff
            if w_node != s:
                scores[w_node] += delta[w_node]
        if progress_every and step % progress_every == 0:
            elapsed = time.time() - t_start
            print(f'  [{label}] {step}/{len(draws)} sources, {elapsed:.0f}s elapsed, '
                  f'~{elapsed / step * (len(draws) - step):.0f}s left')

    scale = total_weight / len(draws)
    meta = {'k': int(len(draws)), 'seed': int(seed),
            'total_weight': total_weight,
            'distinct_sources': int(len(set(draws.tolist()))),
            'seconds': round(time.time() - t_start, 1)}
    return {n: value * scale for n, value in scores.items()}, meta


print('estimator defined')

## 12. הרצת שתי העדשות

כעת אנו מריצים את האומדן פעמיים על הרכיב הקשיר הגדול ביותר:

* **בקרה אחידה** - משקל 1 לכל צומת. זהו betweenness נדגם רגיל, והוא קיים כדי שההשוואה מול עדשת הביקוש תהיה השוואה הוגנת של תפוחים לתפוחים. השוואת עדשת הביקוש ישירות מול העמודה של מחברת 04 הייתה מערבבת שני הבדלים בבת אחת (שקלול שונה *וגם* מדגם מקורות שונה), ולכן אנו מדווחים על שתי ההשוואות ומתייחסים לבקרה כאל ההשוואה העיקרית.
* **משוקללת בביקוש (PROXY)** - משקלי הצמתים הם `demand_weight`. תחנות עם משקל 0 (ללא אוכלוסיית למ"ס, או מחוץ לשיוך הלמ"ס) לעולם אינן נדגמות כמקור ואינן תורמות דבר כיעד, אף שהן עדיין יכולות לשמש כמתווכות ולקבל ציון. זוהי מגבלה אמיתית, וסעיף 15 אומר זאת.

זהו התא היקר: כ-3-8 דקות בסך הכול ב-`K_DEMAND_SOURCES = 400`. שורות התקדמות עם הערכת זמן נותר מודפסות כל `PROGRESS_EVERY` מקורות.

In [ ]:
# --- Run the uniform control lens and the demand-weighted PROXY lens -------
demand_lookup = dict(zip(proxy['stop_id'], proxy['demand_weight'].astype(float)))
demand_weight_lcc = {n: demand_lookup.get(n, 0.0) for n in Gc.nodes}
uniform_weight_lcc = {n: 1.0 for n in Gc.nodes}

covered = sum(1 for v in demand_weight_lcc.values() if v > 0)
print(f'LCC stations with a positive PROXY weight: {covered:,} of {len(demand_weight_lcc):,} '
      f'({covered / len(demand_weight_lcc):.1%})')
print(f'share of the national PROXY weight inside the LCC: '
      f'{sum(demand_weight_lcc.values()):.3%}\n')

print('running the uniform-weight control lens ...')
uniform_scores, uniform_meta = sampled_endpoint_weighted_betweenness(
    Gc, uniform_weight_lcc, K_DEMAND_SOURCES, SEED_UNIFORM,
    progress_every=PROGRESS_EVERY, label='uniform')
print(f'  done in {uniform_meta["seconds"]}s '
      f'({uniform_meta["distinct_sources"]} distinct sources)\n')

print('running the demand-weighted PROXY lens ...')
demand_scores, demand_meta = sampled_endpoint_weighted_betweenness(
    Gc, demand_weight_lcc, K_DEMAND_SOURCES, SEED_DEMAND,
    progress_every=PROGRESS_EVERY, label='demand')
print(f'  done in {demand_meta["seconds"]}s '
      f'({demand_meta["distinct_sources"]} distinct sources)')

## 13. הרכבת הדירוגים וכתיבת טבלת החוזה

שלוש עמודות ציון מצורפות לטבלה אחת, אחת לכל עדשה:

* `topological_betweenness_nb04` - העמודה `approx_betweenness` של מחברת 04, הדירוג הטופולוגי הטהור הקיים בפרויקט;
* `uniform_control_betweenness` - הבקרה מאותו אומדן;
* `demand_weighted_betweenness_proxy` - עדשת הביקוש (שם העמודה נושא את המילה `proxy` במכוון).

כל אחת מהן מומרת לדירוג שבו **1 = הקריטית ביותר**, תוך שימוש בדירוגים ממוצעים עבור תיקו (תחנות רבות מקבלות ציון 0 בדיוק באומדן נדגם, והממוצע מונע מגוש התיקו הזה לייצר סדר מדומה).

`rank_shift = topological_rank - demand_weighted_rank`, כך ש**תזוזה חיובית משמעה שהתחנה חשובה יותר לאחר שנלקחים בחשבון אנשים**, ותזוזה שלילית משמעה שהדירוג הטופולוגי הפריז בהערכתה. קובץ החוזה `tables/demand_weighted_criticality.csv` נושא בדיוק את `stop_id, stop_name, topological_rank, demand_weighted_rank, rank_shift`; כל השאר (ציונים, קואורדינטות, אשכול חברתי-כלכלי, עדשת הבקרה והדירוג שלה) נכתב אל `tables/criticality_lens_detail.csv`.

In [ ]:
# --- Join the three lenses, rank them, write the contract table ------------
lenses = proxy[['stop_id', 'stop_name', 'lat', 'lon', 'region', 'metro',
                'socio_cluster', 'stop_use_count', 'population_served',
                'demand_weight']].copy()

nb04_betweenness = dict(zip(metrics_df['stop_id'],
                            pd.to_numeric(metrics_df['approx_betweenness'],
                                          errors='coerce').fillna(0.0)))
lenses['topological_betweenness_nb04'] = lenses['stop_id'].map(nb04_betweenness).fillna(0.0)
lenses['uniform_control_betweenness'] = lenses['stop_id'].map(uniform_scores).fillna(0.0)
lenses['demand_weighted_betweenness_proxy'] = lenses['stop_id'].map(demand_scores).fillna(0.0)
lenses['in_largest_component'] = lenses['stop_id'].isin(LCC_NODES)

RANK_OF = {
    'topological_betweenness_nb04': 'topological_rank',
    'uniform_control_betweenness': 'uniform_control_rank',
    'demand_weighted_betweenness_proxy': 'demand_weighted_rank',
}
for score_col, rank_col in RANK_OF.items():
    lenses[rank_col] = lenses[score_col].rank(ascending=False, method='average')

lenses['rank_shift'] = lenses['topological_rank'] - lenses['demand_weighted_rank']
lenses['rank_shift_vs_control'] = lenses['uniform_control_rank'] - lenses['demand_weighted_rank']
lenses['label'] = lenses['stop_name'].fillna('').astype(str).str.strip()
lenses['label'] = lenses['label'].where(lenses['label'] != '', lenses['stop_id'])

# --- contract table: exactly the five agreed columns -----------------------
(lenses[['stop_id', 'stop_name', 'topological_rank', 'demand_weighted_rank', 'rank_shift']]
    .sort_values('demand_weighted_rank')
    .to_csv(TABLES / 'demand_weighted_criticality.csv', index=False, encoding='utf-8-sig'))

(lenses.drop(columns=['label'])
       .sort_values('demand_weighted_rank')
       .to_csv(TABLES / 'criticality_lens_detail.csv', index=False, encoding='utf-8-sig'))

print(f'demand_weighted_criticality.csv written ({len(lenses):,} rows)')
print(f'stations with a non-zero demand-weighted score: '
      f'{int((lenses["demand_weighted_betweenness_proxy"] > 0).sum()):,}')
print(f'stations with a non-zero uniform control score: '
      f'{int((lenses["uniform_control_betweenness"] > 0).sum()):,}\n')
print('Top stations under the demand-weighted PROXY lens:')
lenses.nsmallest(TOP_N, 'demand_weighted_rank')[
    ['stop_name', 'region', 'socio_cluster', 'population_served',
     'topological_rank', 'demand_weighted_rank', 'rank_shift']].round(1)

## 14. האם שני הדירוגים מסכימים זה עם זה?

שלוש סטטיסטיקות סיכום לכל זוג עדשות:

* **Spearman rho על פני כל התחנות.** יש לצפות שערך זה יהיה גבוה, ואין לייחס לו משמעות יתרה: לכ-90% מהתחנות יש betweenness קטן אך שונה מאפס, וסדרן היחסי כמעט אינו זז, מה שמנפח את המתאם. זהו רצפה, לא ממצא.
* **Spearman rho על פני איחוד שתי קבוצות ה-top-1000.** צמצום לתחנות שאחת העדשות לפחות רואה בהן חשובות הוא המספר האינפורמטיבי יותר, משום שזוהי האוכלוסייה שמחקר חוסן היה פועל עליה בפועל.
* **חפיפת top-N** עבור N ב-`OVERLAP_SIZES`. "כמה מתוך 50 התחנות שהיית מגן עליהן לפי דירוג טופולוגי עדיין נמצאות בין 50 התחנות שהיית מגן עליהן לאחר שקלול הביקוש." זהו המספר שעל הדוח לצטט.

ההשוואה העיקרית היא **בקרה אחידה מול PROXY הביקוש**, משום ששתיהן מגיעות מאותו אומדן עם אותו `k`, ולכן ההבדל הוא השקלול בלבד. ההשוואה מול מחברת 04 מדווחת אף היא, אך היא סופגת בנוסף את רעש הדגימה של שני אומדנים בלתי תלויים של 300/400 מקורות, ולכן כל אי-הסכמה שם מהווה חסם עליון להשפעה האמיתית של שקלול הביקוש.

In [ ]:
# --- Agreement between the lenses ------------------------------------------
PAIRS = [
    ('uniform_control_betweenness', 'demand_weighted_betweenness_proxy',
     'uniform control (same estimator)', 'demand-weighted PROXY'),
    ('topological_betweenness_nb04', 'demand_weighted_betweenness_proxy',
     'topological nb04', 'demand-weighted PROXY'),
    ('topological_betweenness_nb04', 'uniform_control_betweenness',
     'topological nb04', 'uniform control (same estimator)'),
]

rows = []
for col_a, col_b, name_a, name_b in PAIRS:
    a, b = lenses[col_a], lenses[col_b]
    top_union = set(lenses.nlargest(1000, col_a).index) | set(lenses.nlargest(1000, col_b).index)
    sub = lenses.loc[sorted(top_union)]
    row = {
        'lens_a': name_a,
        'lens_b': name_b,
        'spearman_all_stations': round(float(a.corr(b, method='spearman')), 4),
        'spearman_top1000_union': round(float(sub[col_a].corr(sub[col_b], method='spearman')), 4),
    }
    for n in OVERLAP_SIZES:
        set_a = set(lenses.nlargest(n, col_a)['stop_id'])
        set_b = set(lenses.nlargest(n, col_b)['stop_id'])
        row[f'top{n}_overlap'] = round(len(set_a & set_b) / n, 4)
    rows.append(row)

agreement = pd.DataFrame(rows)
agreement.to_csv(TABLES / 'rank_agreement.csv', index=False, encoding='utf-8-sig')
display(agreement)

ranked = lenses[lenses['in_largest_component']].copy()
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.5))
axes[0].scatter(ranked['uniform_control_rank'], ranked['demand_weighted_rank'],
                s=4, alpha=0.2, color='#2563eb')
lim = float(max(ranked['uniform_control_rank'].max(), ranked['demand_weighted_rank'].max()))
axes[0].plot([1, lim], [1, lim], ls='--', lw=1, color='#334155', label='no change')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].invert_xaxis()
axes[0].invert_yaxis()
axes[0].set_xlabel('Rank under the uniform control lens (1 = most critical)')
axes[0].set_ylabel('Rank under the demand-weighted PROXY lens')
axes[0].set_title('Same estimator, different endpoint weights')
axes[0].legend(loc='lower right')

axes[1].scatter(ranked['topological_rank'], ranked['demand_weighted_rank'],
                s=4, alpha=0.2, color='#dc2626')
lim2 = float(max(ranked['topological_rank'].max(), ranked['demand_weighted_rank'].max()))
axes[1].plot([1, lim2], [1, lim2], ls='--', lw=1, color='#334155', label='no change')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].invert_xaxis()
axes[1].invert_yaxis()
axes[1].set_xlabel('Topological rank (notebook 04 approx_betweenness)')
axes[1].set_ylabel('Rank under the demand-weighted PROXY lens')
axes[1].set_title('Topology vs demand PROXY (also absorbs sampling noise)')
axes[1].legend(loc='lower right')
fig.tight_layout()
fig.savefig(FIGURES / 'rank_rank_scatter.png', dpi=FIG_DPI)
plt.show()

## 15. אילו תחנות זזות - התוצאה בפועל

הפלט המעניין של מחברת זו אינו מקדם מתאם, אלא **רשימת שמות**. סעיף זה מחלץ את התחנות שדירוגן זז יותר מכול בין העדשה הטופולוגית לעדשת ה-PROXY של הביקוש.

שני מנגנוני הגנה מונעים מהרשימה להיות ארטיפקט:

1. רק תחנות **בתוך הרכיב הקשיר הגדול ביותר** ובעלות ציון שונה מאפס בעדשה אחת לפחות כשירות - אחרת גוש התיקו העצום של תחנות בציון אפס היה מייצר "תזוזה" חסרת משמעות.
2. נשקלות רק תחנות המגיעות ל-**2,000 המובילות בעדשה אחת לפחות**. תחנה הזזה מדירוג 24,000 לדירוג 19,000 היא רעש; תחנה הזזה מדירוג 1,900 לדירוג 120 היא התופעה שאנו מחפשים.

`rank_movers.csv` מתעד את שני הכיוונים: **מרוויחות** (`rank_shift` חיובי: תחנות שהמבט הטופולוגי מזלזל בהן, בדרך כלל תחנות בעלות דרגה נמוכה המשוקעות באזורי מגורים צפופים) ו**מפסידות** (`rank_shift` שלילי: צמתים מקושרים היטב במקומות דלילי אוכלוסין - צומתי החלפה בצירים בין-עירוניים, תחנות באזורי תעשייה, צמתים במדבר). יש לזכור ששתי העדשות הן אומדנים נדגמים, ולכן תזוזה של כמה מאות מקומות עבור תחנה בודדת מצויה בתחום הרעש; הדפוס על פני הרשימה כולה הוא הנושא משמעות.

In [ ]:
# --- Biggest gainers and losers of rank ------------------------------------
eligible = lenses[
    lenses['in_largest_component']
    & ((lenses['topological_betweenness_nb04'] > 0)
       | (lenses['demand_weighted_betweenness_proxy'] > 0))
    & ((lenses['topological_rank'] <= 2000) | (lenses['demand_weighted_rank'] <= 2000))
].copy()

gainers = eligible.nlargest(MOVERS_N, 'rank_shift').copy()
losers = eligible.nsmallest(MOVERS_N, 'rank_shift').copy()
gainers['direction'] = 'gains under demand PROXY'
losers['direction'] = 'loses under demand PROXY'

MOVER_COLS = ['direction', 'stop_id', 'stop_name', 'region', 'metro', 'socio_cluster',
              'population_served', 'stop_use_count', 'topological_rank',
              'demand_weighted_rank', 'uniform_control_rank', 'rank_shift',
              'rank_shift_vs_control']
movers = pd.concat([gainers, losers])[MOVER_COLS]
movers.to_csv(TABLES / 'rank_movers.csv', index=False, encoding='utf-8-sig')

print(f'eligible stations for the movers analysis: {len(eligible):,}')
print(f'median |rank shift| among them            : '
      f'{eligible["rank_shift"].abs().median():,.0f} places\n')
print('Biggest GAINERS (topology under-rates them once population is weighted in):')
display(gainers[['stop_name', 'region', 'socio_cluster', 'population_served',
                 'stop_use_count', 'topological_rank', 'demand_weighted_rank',
                 'rank_shift']].round(1).head(TOP_N))
print('Biggest LOSERS (topologically central, but few residents nearby):')
display(losers[['stop_name', 'region', 'socio_cluster', 'population_served',
                'stop_use_count', 'topological_rank', 'demand_weighted_rank',
                'rank_shift']].round(1).head(TOP_N))

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, frame, colour, title in [
    (axes[0], gainers.head(TOP_N), '#16a34a',
     f'Top {TOP_N} rank GAINERS under the demand PROXY'),
    (axes[1], losers.head(TOP_N), '#dc2626',
     f'Top {TOP_N} rank LOSERS under the demand PROXY'),
]:
    frame = frame.sort_values('rank_shift')
    y = np.arange(len(frame))
    ax.barh(y, frame['rank_shift'].astype(float).values, color=colour)
    ax.set_yticks(y)
    ax.set_yticklabels(frame['stop_name'].fillna('').astype(str)
                            .where(frame['stop_name'].notna(), frame['stop_id']).values)
    ax.set_xlabel('Rank shift (topological rank - demand PROXY rank)')
    ax.set_title(title)
fig.tight_layout()
fig.savefig(FIGURES / 'rank_movers.png', dpi=FIG_DPI)
plt.show()

## 16. האם שקלול הביקוש משנה על *מי* הרשת מגנה?

זוהי שאלת השוויוניות שהעלתה מחברת 08, הנשאלת מצד החוסן. אם שקלול הביקוש מקדם באופן שיטתי תחנות באשכולות חברתיים-כלכליים נמוכים, אזי דירוג קריטיות טופולוגי טהור - מהסוג שמפעיל היה משתמש בו כדי לתעדף השקעה ביתירות - מוטה בשקט לרעת שכונות עניות יותר, משום ששכונות אלו צפופות באנשים אך לא בהכרח צפופות במבנה רשת. אם תזוזת הדירוג הממוצעת שטוחה על פני האשכולות, החשש הספציפי הזה אינו נתמך בנתונים.

אנו מדווחים על הממוצע והחציון של `rank_shift` לכל אשכול למ"ס (1 = החלש ביותר, 10 = החזק ביותר), על מספר התחנות לכל אשכול, ועל אופן השתנות חלקו של כל אשכול מתוך 200 המובילות בין שתי העדשות. יש לשים לב לגורם המבלבל לפני קריאת התוצאה: אשכול ו**צפיפות** אוכלוסין מתואמים בישראל, וה-proxy בנוי מצפיפות, ולכן חלק מכל דפוס כאן הוא מכני ולא תגלית על מדיניות תחבורה ציבורית.

In [ ]:
# --- Rank movement by CBS socioeconomic cluster ----------------------------
cluster_frame = lenses.dropna(subset=['socio_cluster']).copy()
cluster_frame['socio_cluster'] = cluster_frame['socio_cluster'].astype(float).round().astype(int)

top_topo = set(lenses.nsmallest(200, 'topological_rank')['stop_id'])
top_demand = set(lenses.nsmallest(200, 'demand_weighted_rank')['stop_id'])
cluster_frame['in_top200_topological'] = cluster_frame['stop_id'].isin(top_topo)
cluster_frame['in_top200_demand'] = cluster_frame['stop_id'].isin(top_demand)

by_cluster = (cluster_frame
              .groupby('socio_cluster')
              .agg(stations=('stop_id', 'size'),
                   mean_rank_shift=('rank_shift', 'mean'),
                   median_rank_shift=('rank_shift', 'median'),
                   mean_population_served=('population_served', 'mean'),
                   top200_topological=('in_top200_topological', 'sum'),
                   top200_demand=('in_top200_demand', 'sum'))
              .reset_index())
by_cluster['top200_change'] = by_cluster['top200_demand'] - by_cluster['top200_topological']
by_cluster = by_cluster.round(2)
by_cluster.to_csv(TABLES / 'rank_shift_by_socioeconomic_cluster.csv',
                  index=False, encoding='utf-8-sig')
display(by_cluster)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colours = ['#16a34a' if v >= 0 else '#dc2626' for v in by_cluster['mean_rank_shift']]
axes[0].bar(by_cluster['socio_cluster'].astype(str), by_cluster['mean_rank_shift'], color=colours)
axes[0].axhline(0, color='#334155', lw=1)
axes[0].set_xlabel('CBS socioeconomic cluster (1 = weakest, 10 = strongest)')
axes[0].set_ylabel('Mean rank shift (+ = promoted)')
axes[0].set_title('Who gains when criticality is weighted by the demand PROXY')

width = 0.4
pos = np.arange(len(by_cluster))
axes[1].bar(pos - width / 2, by_cluster['top200_topological'], width,
            label='top 200, topological', color='#64748b')
axes[1].bar(pos + width / 2, by_cluster['top200_demand'], width,
            label='top 200, demand PROXY', color='#0891b2')
axes[1].set_xticks(pos)
axes[1].set_xticklabels(by_cluster['socio_cluster'].astype(str))
axes[1].set_xlabel('CBS socioeconomic cluster')
axes[1].set_ylabel('Stations in the top 200')
axes[1].set_title('Composition of the top 200 under each lens')
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURES / 'rank_shift_by_socioeconomic_cluster.png', dpi=FIG_DPI)
plt.show()

## 17. עד כמה משנים הפרמטרים השרירותיים של ה-proxy?

שלושה מספרים במחברת זו נבחרו בשיקול דעת ולא נמדדו: רדיוס ה-catchment, אורך הדעיכה ומעריך השירות. תוצאה השורדת רק בהגדרה אחת אינה תוצאה. שני בלוקי רגישות:

* **ברמת המשקלים (זול, `RUN_WEIGHT_SENSITIVITY`).** בניית `demand_weight` מחדש עבור כל צירוף של `RADIUS_GRID` x `SERVICE_EXPONENT_GRID` (אורך הדעיכה משתנה בהתאם לרדיוס) ודיווח מתאם Spearman של כל וריאנט מול משקלי הבסיס. כמה שניות לכל רדיוס.
* **ברמת ה-betweenness (יקר, `RUN_BETWEENNESS_SENSITIVITY`).** הרצה חוזרת של האומדן עצמו עם `SENSITIVITY_K = 150` מקורות עבור שני proxy חלופיים - **אוכלוסייה טהורה** (`SERVICE_EXPONENT = 0`, ללא איבר שירות כלל, כך שהשקלול בלתי תלוי לחלוטין ברשת) ו-**catchment רחב** (רדיוס 1 ק"מ, דעיכה 500 מ') - ודיווח כיצד הדירוג משתווה לעדשת הביקוש הבסיסית. עלות זו היא כ-1-3 דקות נוספות; הווריאנט של אוכלוסייה טהורה הוא החשוב, משום שהוא מראה כמה מדירוג הביקוש מונע בפועל מנפח השירות המתגנב חזרה פנימה.

יש לשים לב שהרצת רגישות ב-`k = 150` רועשת יותר מקו הבסיס ב-`k = 400`, ולכן חלק מכל אי-הסכמה שהיא מראה הוא רעש האומדן ולא השפעת פרמטר אמיתית. הפער בין `k = 150` ל-`k = 400` קובע את הרצפה: שתי הרצות של *אותו* מודל בגדלי מדגם אלה כבר היו נבדלות זו מזו במידת מה.

In [ ]:
# --- Sensitivity of the PROXY to its arbitrary parameters ------------------
baseline_weight = proxy['demand_weight'].to_numpy()
weight_rows = []

if RUN_WEIGHT_SENSITIVITY:
    served_by_radius = {}
    for radius in RADIUS_GRID:
        decay = radius / 2.0
        t0 = time.time()
        served_by_radius[radius] = allocate_population(
            proxy['lat'].to_numpy(), proxy['lon'].to_numpy(),
            proxy['area_population_share'].to_numpy(), radius, decay)
        print(f'  radius {radius:.0f} m / decay {decay:.0f} m: {time.time() - t0:.1f}s')
    for radius in RADIUS_GRID:
        for exponent in SERVICE_EXPONENT_GRID:
            variant = build_demand_weight(served_by_radius[radius],
                                          proxy['stop_use_count'], exponent)
            weight_rows.append({
                'catchment_radius_m': radius,
                'decay_length_m': radius / 2.0,
                'service_exponent': exponent,
                'is_baseline': bool(radius == CATCHMENT_RADIUS_M
                                    and exponent == SERVICE_EXPONENT),
                'spearman_vs_baseline_weight': round(float(
                    pd.Series(variant).corr(pd.Series(baseline_weight), method='spearman')), 4),
                'top200_overlap_vs_baseline': round(float(len(
                    set(pd.Series(variant, index=proxy['stop_id']).nlargest(200).index)
                    & set(pd.Series(baseline_weight, index=proxy['stop_id']).nlargest(200).index)
                ) / 200), 4),
            })
    weight_sensitivity = pd.DataFrame(weight_rows)
    weight_sensitivity.to_csv(TABLES / 'proxy_sensitivity.csv',
                              index=False, encoding='utf-8-sig')
    display(weight_sensitivity)
else:
    weight_sensitivity = pd.DataFrame()
    print('RUN_WEIGHT_SENSITIVITY is False - skipped.')

### 17b. רגישות הדירוג עצמו

התא לעיל מראה רק שה*משקלים* זזים. מה שחשוב הוא האם ה*דירוג* זז. תא זה מריץ מחדש את האומדן על שני proxy חלופיים ומשווה את הדירוגים המתקבלים לעדשת הביקוש הבסיסית, תוך שימוש באותן סטטיסטיקות Spearman / חפיפת top-N כמו בסעיף 14. הוא גם מריץ מחדש את ה-proxy ה**בסיסי** ב-`SENSITIVITY_K` מקורות, מה שמספק את רצפת הרעש: מידת אי-ההסכמה המתקבלת משינוי של גודל המדגם וה-seed בלבד.

In [ ]:
# --- Does the RANKING move when the PROXY definition moves? ----------------
betweenness_sensitivity = pd.DataFrame()
if RUN_BETWEENNESS_SENSITIVITY:
    baseline_series = lenses.set_index('stop_id')['demand_weighted_betweenness_proxy']

    served_pure = proxy['population_served'].to_numpy()
    served_wide = allocate_population(proxy['lat'].to_numpy(), proxy['lon'].to_numpy(),
                                      proxy['area_population_share'].to_numpy(), 1000.0, 500.0)
    VARIANTS = [
        ('noise_floor_baseline_k150', build_demand_weight(
            served_pure, proxy['stop_use_count'], SERVICE_EXPONENT), 101),
        ('pure_population_no_service', build_demand_weight(
            served_pure, proxy['stop_use_count'], 0.0), 102),
        ('wide_catchment_1km', build_demand_weight(
            served_wide, proxy['stop_use_count'], SERVICE_EXPONENT), 103),
    ]

    rows = []
    for name, weight_vector, seed in VARIANTS:
        lookup = dict(zip(proxy['stop_id'], weight_vector))
        weights_lcc = {n: float(lookup.get(n, 0.0)) for n in Gc.nodes}
        print(f'running variant "{name}" at k={SENSITIVITY_K} ...')
        scores, meta = sampled_endpoint_weighted_betweenness(
            Gc, weights_lcc, SENSITIVITY_K, seed,
            progress_every=PROGRESS_EVERY, label=name)
        variant_series = pd.Series(scores).reindex(baseline_series.index).fillna(0.0)
        row = {
            'variant': name,
            'k_sources': meta['k'],
            'seconds': meta['seconds'],
            'spearman_vs_baseline_lens': round(float(
                variant_series.corr(baseline_series, method='spearman')), 4),
        }
        for n in OVERLAP_SIZES:
            row[f'top{n}_overlap_vs_baseline'] = round(float(len(
                set(variant_series.nlargest(n).index) & set(baseline_series.nlargest(n).index)
            ) / n), 4)
        rows.append(row)
        print(f'  {name}: rho = {row["spearman_vs_baseline_lens"]}, '
              f'top50 overlap = {row["top50_overlap_vs_baseline"]}\n')

    betweenness_sensitivity = pd.DataFrame(rows)
    betweenness_sensitivity.to_csv(TABLES / 'betweenness_sensitivity.csv',
                                   index=False, encoding='utf-8-sig')
    display(betweenness_sensitivity)
else:
    print('RUN_BETWEENNESS_SENSITIVITY is False - skipped. '
          'The parameter robustness of the ranking is then untested.')

## 18. JSON סיכום

כל מה שמחברת מאוחרת יותר או הדוח עשויים לרצות לצטט, במילון אחד: פרמטרי ה-proxy, הכיסוי שלו והסתייגויות איכות הנתונים שלו, הגדרות האומדן, סטטיסטיקות ההתאמה בין העדשות, והמרוויחה והמפסידה הגדולות ביותר. הדגל `proxy_is_not_ridership` נכתב כ-`true` במכוון, כך שכל צרכן של קובץ זה השוכח את ההסתייגות יקבל תזכורת מהנתונים עצמם.

In [ ]:
# --- demand_summary.json ---------------------------------------------------
def _agreement(name_a, name_b, key):
    hit = agreement[(agreement['lens_a'] == name_a) & (agreement['lens_b'] == name_b)]
    return float(hit.iloc[0][key]) if len(hit) else None


top_gainer = gainers.iloc[0] if len(gainers) else None
top_loser = losers.iloc[0] if len(losers) else None

summary = {
    'stage': '21_demand_weighted_criticality',
    'proxy_is_not_ridership': True,
    'proxy_description': ('CBS 2021 statistical-area population split across the stops of '
                          'each area, re-allocated with a distance-decayed gravity kernel, '
                          'then multiplied by scheduled service volume. GTFS contains no '
                          'boarding, alighting or ridership data of any kind.'),
    'parameters': {
        'catchment_radius_m': CATCHMENT_RADIUS_M,
        'decay_length_m': DECAY_LENGTH_M,
        'service_exponent': SERVICE_EXPONENT,
        'k_sources_per_lens': K_DEMAND_SOURCES,
        'seed_demand': SEED_DEMAND,
        'seed_uniform_control': SEED_UNIFORM,
        'shortest_paths': 'unweighted hop count, undirected projection, largest component',
    },
    'network': {
        'stops': int(len(lenses)),
        'largest_component_nodes': int(Gc.number_of_nodes()),
        'largest_component_edges': int(Gc.number_of_edges()),
        'largest_component_share': round(Gc.number_of_nodes() / G.number_of_nodes(), 4),
    },
    'proxy_coverage': {
        'cbs_population_total': round(national_population, 0),
        'population_allocated_to_stops': round(allocated, 0),
        'stops_without_cbs_unit': no_unit,
        'stops_whose_unit_has_no_population': no_population,
        'stops_with_zero_demand_weight': int((proxy['demand_weight'] <= 0).sum()),
        'demand_weight_share_of_top_1_percent_of_stops': round(share_top1pct, 4),
        'spearman_population_proxy_vs_service_volume': round(rho_pop_service, 4),
    },
    'agreement': {
        'control_vs_demand_spearman': _agreement(
            'uniform control (same estimator)', 'demand-weighted PROXY',
            'spearman_all_stations'),
        'control_vs_demand_top50_overlap': _agreement(
            'uniform control (same estimator)', 'demand-weighted PROXY', 'top50_overlap'),
        'nb04_vs_demand_spearman': _agreement(
            'topological nb04', 'demand-weighted PROXY', 'spearman_all_stations'),
        'nb04_vs_demand_top50_overlap': _agreement(
            'topological nb04', 'demand-weighted PROXY', 'top50_overlap'),
        'nb04_vs_control_spearman': _agreement(
            'topological nb04', 'uniform control (same estimator)', 'spearman_all_stations'),
        'median_abs_rank_shift_top2000': float(eligible['rank_shift'].abs().median()),
    },
    'biggest_gainer': (None if top_gainer is None else {
        'stop_id': str(top_gainer['stop_id']),
        'stop_name': str(top_gainer['stop_name']),
        'topological_rank': float(top_gainer['topological_rank']),
        'demand_weighted_rank': float(top_gainer['demand_weighted_rank']),
        'rank_shift': float(top_gainer['rank_shift']),
    }),
    'biggest_loser': (None if top_loser is None else {
        'stop_id': str(top_loser['stop_id']),
        'stop_name': str(top_loser['stop_name']),
        'topological_rank': float(top_loser['topological_rank']),
        'demand_weighted_rank': float(top_loser['demand_weighted_rank']),
        'rank_shift': float(top_loser['rank_shift']),
    }),
    'sensitivity_run': {
        'weight_level': bool(RUN_WEIGHT_SENSITIVITY),
        'betweenness_level': bool(RUN_BETWEENNESS_SENSITIVITY),
        'sensitivity_k': SENSITIVITY_K,
    },
    'outputs': sorted(p.name for p in TABLES.glob('*.csv')),
}

with open(STAGE / 'demand_summary.json', 'w', encoding='utf-8') as handle:
    json.dump(summary, handle, ensure_ascii=False, indent=2)

print(json.dumps(summary, ensure_ascii=False, indent=2)[:2500])
print('\nwritten ->', STAGE / 'demand_summary.json')

## 19. מגבלות ה-proxy של הביקוש - יש לקרוא זאת לפני ציטוט של כל מספר שלעיל

סעיף זה ארוך מהרגיל במכוון. המחברת כולה נשענת על proxy, וה-proxy שגוי לפחות בדרכים הבאות.

**1. אוכלוסייה בקרבת תחנה אינה עליות לרכב.** זוהי ההנחה הגדולה ביותר. שתי תחנות בעלות אוכלוסיות תושבים זהות עשויות להיות בעלות מספרי עליות שונים לחלוטין, בהתאם לבעלות על רכב, סביבת ההליכה, שירות מקביל, אזורי תעריף והרגל. דבר בפיד ה-GTFS אינו יכול להבחין ביניהן. `population_served` הוא מדד *חשיפה* - כמה אנשים התחנה קרובה אליהם פיזית - ולא מדד שימוש.

**2. נספרת אוכלוסייה מגורית בלבד; יעדים מתעלמים מהם.** ביקוש אמיתי הוא מטריצה של מוצאים ויעדים. מקומות עבודה, בתי חולים, אוניברסיטאות, קניונים, בסיסי צבא וחופים מייצרים נפחי נסיעה עצומים וכמעט אין בהם תושבים. המודל שלנו משקלל את ה*מוצא* ואת ה*יעד* של מסלול לפי אותו גודל מגורי, ולכן תחנה המשרתת מרכז תעסוקה גדול עם מעט תושבים - בדיוק סוג המקום עם מספר העליות הגבוה ביותר בפועל בשעת השיא של אחר הצהריים - מקבלת משקל חסר באופן שיטתי. זהו הפגם המבני החמור ביותר של ה-proxy, והוא מטה את כל רשימת ה"מפסידות": חלק מהתחנות הללו אינן מוערכות ביתר על ידי הטופולוגיה כלל, הן פשוט משרתות מקומות עבודה ולא בתים.

**3. אין בעלות על רכב, הכנסה או מבנה גילאים.** ביקוש לתחבורה ציבורית לתושב משתנה פי כמה בין שכונות. מכיוון שבעלות על רכב מתואמת עם האשכול החברתי-כלכלי, ניתוח השוויוניות בסעיף 16 מעגלי חלקית: לאזורים באשכול נמוך יש הן תלות גבוהה יותר בתחבורה ציבורית *והן* צפיפות גבוהה יותר, וה-proxy שלנו לוכד רק את מחצית הצפיפות.

**4. אין שעת יום.** ה-proxy הוא מספר סטטי בודד לכל תחנה. לביקוש אמיתי יש שיא בוקר הזורם לעבר מרכזי תעסוקה ושיא ערב הזורם חזרה, וקריטיות בשעה 08:00 היא שאלה שונה מקריטיות בשעה 23:00. מחברות 19 ו-20 מטפלות בממד הזמן עבור הטופולוגיה; שילוב השניים - קריטיות משוקללת בביקוש *לכל חלון זמן* - הוא הצעד הבא הטבעי ואינו נעשה כאן.

**5. פיצול האזור לתחנות הוא נאיבי.** האוכלוסייה מחולקת שווה בשווה בין תחנות אזור סטטיסטי, ולכן שתי תחנות בצדדים מנוגדים של אותו רחוב מקבלות כל אחת מחצית מנה אף שמבחינת הנוסע הן זוג תחנות אחד. אזורים עם תחנות רבות מדללים את עצמם; אזורים עם תחנה אחת מרכזים. גרעין הגרביטציה מרכך זאת אך אינו מתקן זאת.

**6. השיוך ללמ"ס אינו מושלם ומועבר בירושה.** כחמישית מהתחנות הותאמו לאזור הסטטיסטי שלהן לפי *הפוליגון הקרוב ביותר* ולא לפי הכלה (ראו הדפסת איכות השיוך בסעיף 7), ולכמה מאות תחנות אין אזור או אין אוכלוסייה כלל. לתחנות אלו יש משקל ביקוש של 0 בדיוק, כלומר עדשת הביקוש מתייחסת אליהן כאל מקומות שבהם איש אינו מתחיל או מסיים נסיעה. עבור תחנה באזור תעשייה בלתי מאוכלס זה נכון בקירוב; עבור תחנה שהשיוך הפוליגונלי שלה פשוט נכשל זה שגוי.

**7. מסלולים קצרים ביותר נמדדים בקפיצות, ומודל הנסיעה כולו גס.** נוסעים אינם נוסעים לאורך מסלולים קצרים ביותר בגרף סמיכות התחנות; הם נוסעים לאורך *קווים*, עם קנסות החלפה, זמני המתנה והעדפה חזקה לא להחליף כלי רכב. מודל מסלולים המתעלם מעלות ההחלפה סופר ביתר מסלולים המזגזגים בין קווים. מחברת 18 בונה קשתות מבוססות זמן נסיעה, שהיו בסיס טוב יותר, אך ערבוב שינוי זה בהשוואה הנוכחית היה הופך לבלתי אפשרי לייחס הבדל כלשהו לשקלול הביקוש.

**8. שתי העדשות הן אומדנים נדגמים.** עדשת הביקוש משתמשת ב-`K_DEMAND_SOURCES` מקורות נדגמים והעמודה של מחברת 04 השתמשה ב-300. תזוזות דירוג בודדות של כמה מאות מקומות מצויות בתחום הרעש, וזו בדיוק הסיבה שסעיף 15 מגביל את תשומת הלב לתחנות המגיעות ל-2,000 המובילות בעדשה אחת לפחות, ושסעיף 17b מדווח רצפת רעש.

**9. נפח השירות הוא רכיב מעגלי חלקית.** ההכפלה ב-`stop_use_count` מייבאת חזרה מבנה רשת אל תוך משקל שהיה אמור להיות בלתי תלוי בו. וריאנט הרגישות `pure_population_no_service` קיים בדיוק כדי שניתן יהיה לכמת זאת; אם שני הדירוגים מסכימים היטב, איבר השירות אינו זה שעושה את העבודה, ואם לא - עדשת הביקוש הבסיסית היא חלקית דירוג שירות בתחפושת.

**מה היה מתקן זאת:** מספרי עליות לרכב לכל תחנה מנתוני הגבייה האוטומטית של המפעילים (רב-קו), מטריצת מוצא-יעד מבוססת נתוני סלולר או סקר, ומספרי מועסקים לכל אזור סטטיסטי. שלושתם קיימים בישראל; אף אחד מהם אינו נמצא בפיד GTFS ציבורי.

## 20. מסקנות

יש לקרוא אותן יחד עם הטבלאות שב-`outputs/nb/21_demand_weighted_criticality/tables/`. המספרים המדויקים תלויים בתצלום הפיד, בפרמטרי ה-proxy ובזרעי הדגימה.

1. **שקלול ביקוש ניתן ליישום על נתונים אלה, אך רק כ-proxy.** ל-GTFS אין נתוני נסיעות. מה שבנינו הוא אוכלוסייה מגורית בקרבת תחנה, מוחלקת בגרביטציה ונשמרת ברמה הארצית, ובאופן אופציונלי מוכפלת בשירות המתוזמן. כל פריט שמחברת זו כותבת אומר זאת בשמות העמודות שלו, ו-`demand_summary.json` נושא את `proxy_is_not_ridership: true`.

2. **שני הדירוגים מסכימים בצמרת ממש ומתפצלים מתחתיה.** תחנות הגישור הארציות - אלו השוכנות על מסלולים קצרים ביותר כמעט מכל מקום - נותרות קריטיות בכל שקלול שהוא, משום שהן קריטיות מסיבות מבניות שאין להן דבר עם מי שגר בסביבה. נתוני החפיפה של top-50 ו-top-200 ב-`rank_agreement.csv` הם המדד הכן למידת השינוי בתמונה; מתאם Spearman הכולל מנופח על ידי אלפי התחנות בעלות הציון הנמוך שאינן זזות לעולם, ואין לצטטו לבדו.

3. **התוצאה היא רשימת שמות, לא מקדם.** `rank_movers.csv` הוא התוצר. המרוויחות הן בדרך כלל תחנות בעלות דרגה נמוכה המשוקעות בשכונות מגורים צפופות - חסרות ייחוד מבחינה מבנית, אך אנשים רבים גרים סביבן. המפסידות הן בדרך כלל צמתים בצירים בין-עירוניים ותחנות באזורים דלילי אוכלוסין: מרכזיות טופולוגית, ריקות דמוגרפית. האם השקלול מחדש הזה *נכון* תלוי לחלוטין בשאלה האם לדעתך קריטיות משמעה "הרשת נשברת" או "אנשים נותרים תקועים". מחברת זו אינה מכריעה בכך; היא מראה ששתי ההגדרות נותנות תשובות שונות מתחת לשכבה העליונה.

4. **אות השוויוניות בסעיף 16 הוא מרמז במקרה הטוב ומכני חלקית.** כל קידום של שכונות באשכול נמוך נובע בחלקו מכך שה-proxy שלנו *הוא* צפיפות אוכלוסין, ואזורים צפופים בישראל נוטים לאשכולות נמוכים יותר. ראוי לדווח על כך כהשערה - ייתכן שדירוג קריטיות טופולוגי משרת בחסר שכונות עניות צפופות - ולא כממצא מדוד.

5. **הפגם הבודד הגדול ביותר הוא היעדר היעדים.** שקלול שתי נקודות הקצה לפי אוכלוסייה מגורית משמעו שמרכזי תעסוקה, בתי חולים ואוניברסיטאות בלתי נראים כמושכי נסיעות. כל שימוש תפעולי בדירוג זה יידרש למטריצת מוצא-יעד; ללא כזו, יש להתייחס לרשימת ה"מפסידות" בפרט כאל רשימת *שאלות* ולא מסקנות.

6. **הצעד הבא.** לשלב שקלול זה עם גרפי שעות היום של מחברות 19-20 ועם קשתות זמן הנסיעה של מחברת 18: קריטיות משוקללת בביקוש בשעת שיא הבוקר, על רשת שבה המרחק נמדד בדקות, היא הקירוב הטוב ביותר שמחקר המבוסס על GTFS בלבד יכול להגיע אליו לשאלה שמפעיל באמת שואל.